In [21]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm

# CONFIG

In [22]:
SEED = 42
NFOLDS = 5
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 5*2
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [23]:
# Set seeds
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [25]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [26]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [27]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [28]:
test_df = pd.read_csv(test_path)

# Get augmented data from both df and test_df
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with augmented data
augmented_texts =  augmented_train[0] + augmented_test[0]
augmented_labels =  augmented_train[1] + augmented_test[1]


# Create new augmented dataframe
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'label': 'mean'   # Take mean of labels
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

augmented_df.head()

Before:(8156, 2)
After: (996, 2)


,text,label,rule,body,rule_id
0,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...","\n\nIf you have some free time on your hands, ...",0
1,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n\nSD | [ English Stream 1 Arsenal vs Tottenh...,0
2,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n**HD** ENG [ 1080P HD Amazing] :- [USTREAM E...,0
3,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\nHD www.stremstar.com/ch3.php ENGLISH MOBILL ...,0
4,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...","\nHey, you do not want to cmprar my awp asiimo...",0


In [29]:
# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [30]:
# print(df.head())
# print(df.columns.tolist())

In [31]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

In [32]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [33]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [34]:
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]  # Assuming this is already on CPU as integers
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        # Only compute AUC if we have both positive and negative samples for this rule
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            # If only one class present, we can't compute AUC
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    # print(rule_aucs,'Rule_AUC')
    return avg_auc_per_rule, val_loss, preds

In [35]:
from transformers import get_linear_schedule_with_warmup

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df['rule_id']= df.rule.str.lower().map(rule_map)
    filtered_df= df.query('text not in @augmented_df.text.values')
    print(filtered_df.shape)
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        print('--------- ','FOLD: ',fold,' --------')
        all_preds = []
        val_ds = JigsawDataset(
            filtered_df['text'].tolist(), 
            filtered_df['label'].tolist(), 
            filtered_df['rule_id'].tolist(), 

            tokenizer, MAX_LEN
        )
    
        train_ds = JigsawDataset(
            augmented_df.iloc[tr_idx]['text'].tolist(), 
            augmented_df.iloc[tr_idx]['label'].tolist(), 
            augmented_df.iloc[tr_idx]['rule_id'].tolist(), 
            
            tokenizer, MAX_LEN,
        )
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        # Initialize classification model and load MLM pre-trained weights
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        for name, param in model.named_parameters():
            if name.startswith('base.embedding'):
                param.requires_grad = False
                # print(name)
                
        print('Trainable Params: ',sum(i.numel() for i in model.parameters() if i.requires_grad))
       
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5,eps=1e-6)
        total_steps= EPOCHS*len(train_loader)
        warmup_steps= 0.1*total_steps
        scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps,
            )

    
        best_auc=0
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch+1}/{EPOCHS}")
            loss = train_one_epoch(model, train_loader, optimizer, scheduler)
            val_auc, val_loss ,val_preds = validate(model, val_loader)
            
                        
            print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"model_fold{fold}_auc.bin")
    
        all_preds.append(pd.Series(val_preds))

(883, 12)
---------  FOLD:  0  --------
Trainable Params:  85449985
Epoch 1/10


100%|██████████| 25/25 [00:18<00:00,  1.35it/s]


Loss: 0.6966, Val Loss: 0.7120, Val AUC: 0.5806
Epoch 2/10


100%|██████████| 25/25 [00:18<00:00,  1.32it/s]


Loss: 0.6471, Val Loss: 0.6844, Val AUC: 0.6084
Epoch 3/10


...

100%|██████████| 25/25 [00:21<00:00,  1.18it/s]


Loss: 0.0236, Val Loss: 1.5257, Val AUC: 0.6828
Epoch 8/10


100%|██████████| 25/25 [00:20<00:00,  1.19it/s]


Loss: 0.0142, Val Loss: 1.6499, Val AUC: 0.6826
Epoch 9/10


100%|██████████| 25/25 [00:21<00:00,  1.19it/s]


what to do at test time, dont take examples from test set into val.

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    
    all_truths=[]
    all_rules=[]
    all_preds=[]
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        
        wts=torch.load(f"model_fold{fold}_auc.bin", map_location=DEVICE)
        wts= {k.replace('module.',''):v for k,v in wts.items()}
                
        model.load_state_dict(wts)
        model.eval()
        val_ds = JigsawDataset(
                augmented_df.iloc[val_idx]['text'].tolist(), 
                augmented_df.iloc[val_idx]['label'].tolist(), 
                augmented_df.iloc[val_idx]['rule_id'].tolist(), 
    
                tokenizer, MAX_LEN
            )
        all_truths.append(augmented_df.iloc[val_idx]['label'].apply(lambda x:x>=.5).astype('float'))
        all_rules.append(augmented_df.iloc[val_idx].rule)
            
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
        
        _,_,val_preds= validate(model,val_loader)
                
        all_preds.append(pd.Series(val_preds))
                
    
    preddf= pd.DataFrame(columns=['preds','truths','rule'])
    preddf.preds=pd.concat(all_preds,ignore_index=True)
    preddf.rule= pd.concat(all_rules,ignore_index=True)
    preddf.truths= pd.concat(all_truths,ignore_index=True)
    
    print(preddf.groupby('rule').apply(lambda group: roc_auc_score(group['truths'],group['preds'])))

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_preds = []
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        wts=torch.load(f"model_fold{fold}_auc.bin", map_location=DEVICE)
        wts= {k.replace('module.',''):v for k,v in wts.items()}
            
        model.load_state_dict(wts)
        model.eval()
    
        test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test),[0]*len(df_test), tokenizer, MAX_LEN)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_preds.append(fold_preds)

In [ ]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    final_preds = np.mean(test_preds, axis=0)
    sample["rule_violation"] = final_preds
    sample.to_csv("submission.csv", index=False)
    print("✅ Submission saved as submission.csv")
else:
    !touch submission.csv
    
!head -n 4 submission.csv